In [19]:
from youtube_transcript_api import YouTubeTranscriptApi , TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate

In [15]:
video_id = 'MdeQMVBuGgY'
ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(video_id, languages=["en"])

# Flatten to text
transcript = " ".join(snippet.text for snippet in transcript)
print(transcript)

Mr. Malia is out of the country. Sicker Baron Vijay Malia who find himself in a raging 9,000 cr loan default scam. Welcome on figuring out sir. Why are you doing this after 9? In India public opinion is very quickly formed and influenced by the media pasa loot. Hi, I'm Vijay Malian. So it's not just huge, it is and nothing else comes close. King Fisher is a class apart. VJ Mallay is you were the biggest airline in the country. Welcome aboard King Fisher Airlines India's biggest airline going to be there amongst the best airlines in the world. What went wrong? So I went to Shri Pranabokuji then finance minister of India. I have to downsize this aine. I was told not to downsize King Fisher at that time. You continue, banks will support you. That is how it all started. Kingfisher Airlines has been forced to suspend all of its flights. Kingfisher Airlines has been struggling. At the time when you asked loan, the company was not doing that great. Please never forget I gave my personal guara

In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])
print(len(chunks))
print(chunks[20])

297
page_content='and when you finish at 10:30, you go and work a full day at the office as a management trainee. I was put on a stipend of 400 rupees a month and that 400 rupees never increased right the way my time through college. So that's all I got 400 rupees a month and was working in those days fips and company limited old courthouse street in Kolkata every day after college. Aren't there stories that you used to go to college in a very fancy car and you removed the silencer just to show the the noise of your car, the loudness of your car and you were always a show off, weren't you? I was no showoff at all. And the fancy car that I have was a secondhand standard herald. So where are these stories coming from? I don't know. People's narratives, people's perceptions. That's why it's important to'


In [26]:
embeddings = OllamaEmbeddings(model = 'nomic-embed-text')
vector_store = Chroma.from_documents(
    documents= chunks,
    embedding=embeddings
)


Step 2 - retriever

In [32]:
retriever = vector_store.as_retriever(search_type= 'mmr',
                                      search_kwargs ={'k':5})


In [33]:
res = retriever.invoke("what fraud did vijay did")
for i in res:
    print(i.page_content)

fraud or people who are businesses have failed are fraud because they've taken money from the banks which is a common taxpayers money which is common people's money. So if you take common people's money and you're not able to repay it then you're considered wrong. But if in your case if they have actually recovered more than 2.5x of it or probably somewhere around that number why are they still behind you? Because as I said, have you ever heard of any bank not rendering an account to a borrower or a guaranter? Despite 15 reminders, the same banks have not given me a statement of account. Not one. Have you ever heard of such conduct anywhere in the world? For the first time, it is through the statement in parliament of the finance minister and the printed report of the ministry of finance that they have shown 14,100 crores recovered from me. The government have the banks have not rendered a statement of account. That is why I have gone to the Karnataka High Court asking the court to
dow

step 3 - augmentation

In [ ]:
model = ChatOllama(model='qwen3:8b', reasoning=False)

In [35]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

      {context}
      Question: {question}""",
      input_variables=['context','question']
)

In [60]:
query= 'what is the host of the podcast'
retrieved_docs = retriever.invoke(query)


In [61]:
context = '\n\n'.join(i.page_content for i in retrieved_docs)

In [62]:
final_prompt = prompt.invoke({'question':query, "context":context})

step - 4, Generation

In [63]:
res = model.invoke(final_prompt)
print(res.content)

The host of the podcast is not explicitly mentioned in the provided transcript.


now doiing this using the chains

In [49]:
from langchain_core.runnables import RunnableBranch, RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [46]:
parser = StrOutputParser()

In [50]:
def format_docs(retrieved_docs):
    context = '\n\n'.join(i.page_content for i in retrieved_docs)
    return context



In [56]:
parallel = RunnableParallel({
    'context':  retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})


In [57]:
chain2 = prompt | model | parser

final_chain = parallel | chain2 

In [69]:
query= 'summarize the whole video, dont leave any topics'
res = final_chain.invoke(query)
print(res)

The video discusses the situation of Vijay Malia, a prominent figure in the Indian corporate sector, who is currently in legal trouble in India. The narrative begins with a critique of the media's role in shaping public perception, particularly focusing on how one television anchor's comments sparked a media frenzy against Malia. The speaker emphasizes the importance of understanding all sides of a story and not taking sides, urging viewers to approach the situation with sincerity and curiosity.

The summary of the series of events includes Malia's departure from India, which was initially planned as a scheduled meeting in Geneva. After a month, he was suspended, his passport was revoked, and he was stuck in India. The extradition process began, and he was arrested twice but granted bail. The case went to trial, with CBI, ED, and the government finding him guilty, although he maintains that no one has officially found him guilty.

The speaker highlights the Supreme Court's order for Ma